In [3]:
import pandas as pd
from pathlib import Path
import joblib

In [4]:
# Load the latest timestamped model.
model_folder = Path("models")
model_files = sorted(model_folder.glob("xgb_chl_model_*.joblib"), reverse=True)

if not model_files:
    raise FileNotFoundError(f"No saved model found in {model_folder.resolve()}")

model_path = model_files[0]
model_bundle = joblib.load(model_path)

In [5]:
# Load the latest timestamped test CSV.
split_folder = Path("data") / "split data"
test_files = sorted(split_folder.glob("latest_data_test_*.csv"), reverse=True)

if not test_files:
    raise FileNotFoundError(f"No test CSV found in {split_folder.resolve()}")

test_path = test_files[0]
test = pd.read_csv(test_path)

print(f"Model file: {model_path.name}")
print(f"Model saved at: {model_bundle['saved_at']}")
print(f"Test file: {test_path.name}")
print(f"Test samples: {len(test)}")

Model file: xgb_chl_model_20261114_155638.joblib
Model saved at: 2026-11-14T15:56:38+08:00
Test file: latest_data_test_20261114_153936.csv
Test samples: 4


In [6]:
# Match the training feature order and apply the saved scaler.
TARGET = "moisture_chl"

X_test = test.loc[:, model_bundle["input_features"]]
y_test = test[TARGET]
X_test_scaled = model_bundle["scaler"].transform(X_test)
model = model_bundle["model"]
y_pred = model.predict(X_test_scaled)

predictions = pd.DataFrame({
    "TX Label": test["tlabel"],
    "Actual CHL (%)": test["moisture_chl"],
    "Predicted CHL (%)": y_pred,
})

print("\nActual vs predicted")
print(predictions.round(4).to_string(index=False))


Actual vs predicted
TX Label  Actual CHL (%)  Predicted CHL (%)
      T1            3.45             3.5724
      T6            3.30             4.2814
     T12            4.10             2.9660
      T2            0.90             2.0606


In [9]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def percentage_error(y_test, y_pred):
    y_test = np.asarray(y_test, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    errors = np.full(y_test.shape, np.nan, dtype=float)
    np.divide(
        100 * np.abs(y_pred - y_test),
        np.abs(y_test),
        out=errors,
        where=y_test != 0,
    )
    return errors

# Evaluate the predictions against measured CHL.
r2 = r2_score(y_test, y_pred) if len(y_test) >= 2 else np.nan
errors = percentage_error(y_test, y_pred)
absolute_error = np.abs(y_pred - y_test)
valid_errors = errors[~np.isnan(errors)]

metrics = pd.DataFrame({
    "r2": [r2],
    "rmse": [np.sqrt(mean_squared_error(y_test, y_pred))],
    "mae": [mean_absolute_error(y_test, y_pred)],
    "mape_percent": [valid_errors.mean() if valid_errors.size else np.nan],
})

predictions = pd.DataFrame({
    "TX Label": test["tlabel"],
    "test CHL (%)": y_test,
    "pred CHL (%)": y_pred,
    "Absolute Error (percentage points)": absolute_error,
    "Error (%)": errors,
})

print("\nModel performance")
print(metrics.round(4).to_string(index=False))
print("\nActual vs predicted")
print(predictions.round(4).to_string(index=False))


Model performance
    r2   rmse    mae  mape_percent
0.3876 0.9501 0.8496        47.474

Actual vs predicted
TX Label  test CHL (%)  pred CHL (%)  Absolute Error (percentage points)  Error (%)
      T1          3.45        3.5724                              0.1224     3.5466
      T6          3.30        4.2814                              0.9814    29.7396
     T12          4.10        2.9660                              1.1340    27.6581
      T2          0.90        2.0606                              1.1606   128.9516


In [10]:
from sklearn.inspection import permutation_importance

FEATURES = model_bundle["input_features"]

# Measure the signed change in test MAE after shuffling each feature.
result = permutation_importance(
    model,
    X_test_scaled,
    y_test,
    n_repeats=30,
    random_state=42,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
)

mean_mae_change = result.importances_mean
positive_increases = mean_mae_change.clip(min=0)
total_increase = positive_increases.sum()

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean MAE change (percentage points)": mean_mae_change,
    "Importance (%)": (
        100 * positive_increases / total_increase if total_increase > 0 else 0.0
    ),
    "Action": np.where(mean_mae_change < 0, "Candidate for removal", ""),
}).sort_values("Mean MAE change (percentage points)", ascending=False)

print("\nPermutation feature importance")
print(importance.to_string(
    index=False,
    formatters={
        "Mean MAE change (percentage points)": "{:+.4f}".format,
        "Importance (%)": "{:.2f}%".format,
    },
))
print("\n\033[32mDone processing ...\033[0m\n")


Permutation feature importance
                       Feature Mean MAE change (percentage points) Importance (%)                Action
                   BV_moisture                             +0.0637         27.20%                      
                       BV_2FAL                             +0.0564         24.10%                      
                         BV_N2                             +0.0391         16.71%                      
BV_percent_relative_saturation                             +0.0365         15.57%                      
                 humidity_surr                             +0.0149          6.37%                      
                       BV_C2H6                             +0.0084          3.57%                      
                     BV_sludge                             +0.0047          2.00%                      
                   BV_oil_temp                             +0.0031          1.31%                      
                        BV_bdv  